# EAPS 507 — Pandas for Earth Science Data Manipulation
## A 2-hour guided in-class tutorial using Indiana geographic features

**Dataset:** `IN_Features_20140204.txt`

### Learning objectives
By the end of this activity, you should be able to:

1. load a non-comma-delimited text file with `pandas`;
2. inspect a `DataFrame` and diagnose its structure;
3. select columns and filter observations with Boolean conditions;
4. identify and handle missing values deliberately;
5. summarize categorical and numerical data;
6. use `groupby()` and `agg()` to answer scientific questions;
7. sort data and create derived variables;
8. convert text dates into datetime objects; and
9. combine several Pandas operations to answer a question you formulate yourself.

### How this notebook works
This is **not** a copy-and-paste exercise. Each section follows the same cycle:

> **Explain → instructor example → predict/test → your turn → interpret**

Some cells are complete. Others contain blanks (`...`) for you to complete. Before running a **Predict first** cell, write down what you expect to happen.

**Suggested pace:** Sections 1–3 (45 min), Sections 4–6 (45 min), Sections 7–8 + final challenge (30 min).

## 0. Setup and the dataset — about 5 minutes

The file contains named geographic features and attributes such as feature class, state, county, latitude/longitude, elevation, map name, and dates. It is a **pipe-delimited** text file, so fields are separated by `|` rather than commas.

The goal today is not to memorize Pandas syntax. The goal is to learn how to move from a **data question** to an appropriate sequence of Pandas operations.

In [ ]:
import pandas as pd

# This option makes wide tables easier to inspect in Jupyter.
pd.set_option("display.max_columns", 25)

In [ ]:
# read_csv() can read many delimited text files, not only .csv files.
# sep="|" tells Pandas that the vertical bar separates the columns.

df = pd.read_csv("IN_Features_20140204.txt", sep="|")

df.head()

,FEATURE_ID,FEATURE_NAME,FEATURE_CLASS,STATE_ALPHA,STATE_NUMERIC,COUNTY_NAME,COUNTY_NUMERIC,PRIMARY_LAT_DMS,PRIM_LONG_DMS,PRIM_LAT_DEC,PRIM_LONG_DEC,SOURCE_LAT_DMS,SOURCE_LONG_DMS,SOURCE_LAT_DEC,SOURCE_LONG_DEC,ELEV_IN_M,ELEV_IN_FT,MAP_NAME,DATE_CREATED,DATE_EDITED
0,403294,Brouilletts Creek,Stream,IN,18,Whitley,183,393617N,0872305W,39.604762,-87.384782,393919N,0874800W,39.655230,-87.799900,144.0,472.0,New Goshen,01/15/1980,11/19/2010
1,404056,Beaver Creek,Stream,IL,17,Iroquois,75,405811N,0874850W,40.969755,-87.813927,410159N,0872342W,41.033090,-87.395030,186.0,610.0,L'Erable,01/15/1980,NaN
2,406202,Clear Creek,Stream,IL,17,Edgar,45,392645N,0873001W,39.445869,-87.500300,393514N,0874118W,39.587257,-87.688360,146.0,479.0,Dennison,01/15/1980,NaN
3,406288,Coal Creek,Stream,IL,17,Edgar,45,393829N,0873006W,39.641425,-87.501687,393855N,0873750W,39.648646,-87.630580,151.0,495.0,Saint Bernice,01/15/1980,NaN
4,406345,Cole Creek,Stream,IL,17,Iroquois,75,403756N,0873331W,40.632259,-87.558636,403756N,0873331W,40.632259,-87.558636,207.0,679.0,Darrow,01/15/1980,NaN


### Student-led test 0
Without running another command yet, inspect the first five rows above.

1. Name **two categorical** variables.
2. Name **two numerical** variables.
3. Identify one variable that appears to contain missing values.
4. Why would `pd.read_csv(..., sep="|")` work even though the filename ends in `.txt`?

Discuss your answers with a neighbor for 2 minutes.

---
## 1. Inspecting a DataFrame — about 15 minutes

Before manipulating a dataset, learn its structure.

Useful attributes/methods:

- `df.head()` — first rows
- `df.tail()` — last rows
- `df.shape` — `(number of rows, number of columns)`
- `df.columns` — column labels
- `df.dtypes` — inferred data type of each column
- `df.info()` — compact summary including non-null counts and data types

Notice that `shape`, `columns`, and `dtypes` are **attributes**, while `head()` and `info()` are **methods**.

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

### Predict first
What do you expect Pandas to report for `ELEV_IN_M`, `FEATURE_NAME`, and `DATE_CREATED`?

Write your prediction before running the next cell.

In [ ]:
df[["ELEV_IN_M", "FEATURE_NAME", "DATE_CREATED"]].dtypes

### Why is `DATE_CREATED` not automatically a date?
Pandas reads many text fields as `object` unless we explicitly tell it how to interpret them. Later, we will convert the date columns.

### Student-led test 1
Complete the cells below. Do not ask the instructor for the answer until you have tried the documentation/help system or discussed it with a neighbor.

In [ ]:
# TEST 1A
# Display the LAST 7 records in the dataset.
# Hint: a method introduced above does this.

...

In [ ]:
# TEST 1B
# Use one Pandas command to display the number of non-null observations
# and data type for every column.

...

**Interpretation:** Which columns immediately look like they may require special handling? Why?

---
## 2. Selecting columns and rows — about 15 minutes

A DataFrame is two-dimensional. We often need only part of it.

### Selecting one column
`df["COLUMN"]` returns a Pandas `Series`.

### Selecting several columns
Pass a **list of column names**: `df[["A", "B"]]`.

### `.loc[]`
`.loc[rows, columns]` is a flexible label-based selector.

In [ ]:
# One column -> Series
feature_names = df["FEATURE_NAME"]
print(type(feature_names))
feature_names.head()

In [ ]:
# Several columns -> DataFrame
location = df[["FEATURE_NAME", "COUNTY_NAME", "PRIM_LAT_DEC", "PRIM_LONG_DEC"]]
location.head()

In [ ]:
# .loc[row selection, column selection]
# Here ':' means all rows.
df.loc[:, ["FEATURE_NAME", "FEATURE_CLASS", "ELEV_IN_M"]].head()

### Student-led test 2
Create a DataFrame named `geo` containing only:

`FEATURE_NAME`, `FEATURE_CLASS`, `COUNTY_NAME`, `PRIM_LAT_DEC`, `PRIM_LONG_DEC`, and `ELEV_IN_M`.

Then display rows 10 through 15 using `.loc[]`.

**Test yourself:** Is the end label included when using `.loc[10:15]`? Determine this from your output rather than guessing.

In [ ]:
columns_needed = [
    # Complete this list
]

geo = df[columns_needed]

# Display rows 10 through 15
...

---
## 3. Boolean filtering — about 20 minutes

Filtering asks Pandas to keep only rows satisfying a condition.

For example, `df["FEATURE_CLASS"] == "Stream"` does **not** immediately return streams. It first produces a Boolean Series of `True` and `False` values. That Boolean mask can then be used to select rows.

In [ ]:
is_stream = df["FEATURE_CLASS"] == "Stream"
is_stream.head(10)

In [ ]:
streams = df[is_stream]
streams[["FEATURE_NAME", "COUNTY_NAME", "ELEV_IN_M"]].head()

### Predict first
What will the expression below return: a number, a Series, or a DataFrame?

```python
(df["ELEV_IN_M"] > 300)
```

Make your prediction, then test it.

In [ ]:
df["ELEV_IN_M"] > 300

### Combining conditions
Use:

- `&` for AND
- `|` for OR
- `~` for NOT
- `.isin([...])` when matching several allowed values

Put each condition in parentheses.

In [ ]:
# Indiana streams above 250 m
high_in_streams = df[
    (df["STATE_ALPHA"] == "IN") &
    (df["FEATURE_CLASS"] == "Stream") &
    (df["ELEV_IN_M"] > 250)
]

high_in_streams[["FEATURE_NAME", "COUNTY_NAME", "ELEV_IN_M"]].head()

In [ ]:
# isin() is cleaner than writing many OR statements.
water_features = df[df["FEATURE_CLASS"].isin(["Stream", "Lake", "Reservoir", "Canal"])]
water_features.head()

### Student-led test 3A — basic filtering
Create `lakes` containing only records whose `FEATURE_CLASS` is `"Lake"`.

Then determine how many rows it contains. **Do not use a number supplied by the instructor. Let the data answer the question.**

In [ ]:
lakes = ...

# How many?
...

### Student-led test 3B — combine conditions
Find all records that:

1. are in Indiana,
2. are either `Lake` or `Reservoir`, and
3. have elevation greater than 250 m.

Store the result as `high_water`. Display only feature name, class, county, and elevation.

In [ ]:
high_water = df[
    ...
]

...

### Check your reasoning
Explain why this expression can be dangerous:

```python
df["FEATURE_CLASS"] == "Lake" | df["FEATURE_CLASS"] == "Reservoir"
```

What role do parentheses play when combining Pandas Boolean conditions?

---
## 4. Missing data — about 15 minutes

Real Earth-science datasets are rarely complete.

Useful tools:

- `.isna()` — identify missing values
- `.notna()` — identify non-missing values
- `.isna().sum()` — count missing values by column
- `.dropna()` — remove observations with missing values
- `.fillna()` — replace missing values

**Important:** missing-data treatment should follow the scientific question. Do not automatically drop every row containing any missing value.

In [ ]:
missing = df.isna().sum()
missing.sort_values(ascending=False)

### Predict first
Consider:

```python
df.dropna()
```

Would you expect this to retain most of the dataset or remove a large fraction? Use the missing-value summary above to justify your prediction **before** running it.

In [ ]:
# Now test the prediction.
print("Original rows:", len(df))
print("Rows after dropna() on ALL columns:", len(df.dropna()))

In [ ]:
# A targeted approach: require elevation only.
elevation_data = df.dropna(subset=["ELEV_IN_M"])

print("Original rows:", len(df))
print("Rows with elevation:", len(elevation_data))
print("Rows removed:", len(df) - len(elevation_data))

### Student-led test 4
Calculate the **percentage** of records with missing source latitude (`SOURCE_LAT_DEC`).

You know that a Boolean Series behaves like 1 for `True` and 0 for `False` when averaged. Try to use that idea.

Then answer: would filling missing source latitude with `0` be scientifically sensible? Why or why not?

In [ ]:
missing_source_pct = ...
print(f"Missing source latitude: {missing_source_pct:.1f}%")

---
## 5. Counting, grouping, and aggregation — about 25 minutes

This is where Pandas becomes especially useful for scientific analysis.

### `value_counts()`
Use this when you want frequencies of unique categories.

In [ ]:
df["FEATURE_CLASS"].value_counts().head(10)

### Student-led test 5A
Which five counties contain the largest number of records **for Indiana records only**?

You have already used filtering and `value_counts()`. Combine them.

In [ ]:
indiana = df[df["STATE_ALPHA"] == "IN"]

# Your code here
...

### `groupby()`
Conceptually, `groupby()` performs **split → apply → combine**:

1. split rows into groups;
2. apply a calculation to each group;
3. combine the group results.

Here we ask: *What is the mean elevation for each county?*

In [ ]:
county_elevation = (
    indiana.groupby("COUNTY_NAME")["ELEV_IN_M"]
           .mean()
)

county_elevation.head()

### Predict first
What does each row in `county_elevation` represent?

Now predict what changing `.mean()` to `.max()` would mean scientifically.

In [ ]:
indiana.groupby("COUNTY_NAME")["ELEV_IN_M"].max().head()

### `agg()` — several summaries at once
`agg()` lets us apply multiple summary functions to each group.

In [ ]:
feature_elevation = (
    indiana.groupby("FEATURE_CLASS")["ELEV_IN_M"]
           .agg(["count", "mean", "median", "min", "max"])
)

feature_elevation.head(10)

### Student-led test 5B
Using `groupby()` and `agg()`, calculate the following elevation statistics **by county**:

- count
- mean
- standard deviation
- minimum
- maximum

Hint: the aggregation name for standard deviation is `"std"`.

In [ ]:
county_stats = ...
county_stats.head()

### Student-led test 5C — make a scientific comparison
Compare elevation statistics for only `Stream`, `Lake`, and `Summit` features.

Do this in two stages:

1. filter the rows;
2. group by `FEATURE_CLASS` and calculate `count`, `mean`, `min`, and `max`.

Which class has the highest mean elevation?

In [ ]:
selected = ...

comparison = ...

comparison

---
## 6. Sorting and creating new variables — about 15 minutes

### `sort_values()`
Sorting is often the last step after calculating a quantity.

In [ ]:
# Ten highest-elevation Indiana features
highest = (
    indiana.sort_values("ELEV_IN_M", ascending=False)
           [["FEATURE_NAME", "FEATURE_CLASS", "COUNTY_NAME", "ELEV_IN_M"]]
           .head(10)
)

highest

### Creating a derived column
Scientific datasets often contain variables from which we calculate new quantities.

This dataset already contains elevation in meters and feet. We can use that redundancy to test a conversion.

In [ ]:
# Make a copy before adding experimental/derived columns.
work = indiana.copy()

work["ELEV_FT_CALCULATED"] = work["ELEV_IN_M"] * 3.28084
work["ELEV_FT_DIFFERENCE"] = work["ELEV_IN_FT"] - work["ELEV_FT_CALCULATED"]

work[["ELEV_IN_M", "ELEV_IN_FT", "ELEV_FT_CALCULATED", "ELEV_FT_DIFFERENCE"]].head()

### Student-led test 6
Use Pandas to answer both questions:

1. What is the largest **absolute** difference between the supplied feet elevation and your calculated value?
2. Why might the difference not be exactly zero?

Hint: a Pandas Series has an `.abs()` method.

In [ ]:
max_abs_difference = ...
max_abs_difference

---
## 7. Dates in Pandas — about 10 minutes

`DATE_CREATED` was initially read as text. To perform date-aware operations, convert it with `pd.to_datetime()`.

`errors="coerce"` means values that cannot be interpreted as dates become `NaT` (Not-a-Time) rather than stopping the program.

In [ ]:
work["DATE_CREATED"] = pd.to_datetime(work["DATE_CREATED"], errors="coerce")
work["DATE_EDITED"] = pd.to_datetime(work["DATE_EDITED"], errors="coerce")

work[["DATE_CREATED", "DATE_EDITED"]].dtypes

In [ ]:
# The .dt accessor exposes datetime components.
work["YEAR_CREATED"] = work["DATE_CREATED"].dt.year

work[["FEATURE_NAME", "DATE_CREATED", "YEAR_CREATED"]].head()

### Student-led test 7
Answer these with Pandas:

1. What is the earliest creation date in the dataset?
2. What is the latest creation date?
3. How many records were created in each year?
4. Which creation year has the most records?

For #3, try `value_counts()` on `YEAR_CREATED` and then decide whether sorting by **count** or by **year** better answers the question you are asking.

In [ ]:
earliest = ...
latest = ...

print("Earliest:", earliest)
print("Latest:", latest)

# Continue your analysis below
...

---
# 8. Final challenge — about 20 minutes
## From Pandas syntax to data analysis

Work individually or in pairs. You may use earlier sections of this notebook, but your final code should run from top to bottom.

Using **Indiana records only**, answer the following:

1. What are the five most common feature classes?
2. Which five counties contain the greatest number of features?
3. For `Stream`, `Lake`, and `Summit`, calculate the number of observations and mean, minimum, and maximum elevation.
4. Identify the 10 highest-elevation features and report their name, feature class, county, and elevation.
5. What percentage of records are missing source coordinates?
6. **Develop one additional question that can be answered with this dataset.**

For Question 6, include:

- your question;
- the Pandas operations you chose;
- your result; and
- a 2–3 sentence interpretation.

### Constraint
For at least one question, your solution must combine **three or more Pandas operations** (for example: filter → group → aggregate → sort).

In [ ]:
# FINAL CHALLENGE — Question 1
# Five most common feature classes in Indiana

...

In [ ]:
# FINAL CHALLENGE — Question 2
# Five counties with the greatest number of features

...

In [ ]:
# FINAL CHALLENGE — Question 3
# Stream/Lake/Summit elevation summary

...

In [ ]:
# FINAL CHALLENGE — Question 4
# Ten highest-elevation Indiana features

...

In [ ]:
# FINAL CHALLENGE — Question 5
# Percentage missing source coordinates
# Decide how you will define "missing source coordinates".
# State that definition in a comment before your calculation.

...

### FINAL CHALLENGE — Question 6: Your own question

**My question:**  

**Why this question is useful/interesting:**  

**Pandas operations I expect to need:**

In [ ]:
# Write your analysis for Question 6 here.

...

**Interpretation (2–3 sentences):**

---
## Exit ticket — 5 minutes
Answer without copying definitions from above.

1. What is the difference between `value_counts()` and `groupby()`?
2. Why can `dropna()` be dangerous when used without specifying a subset?
3. What does a Boolean mask contain?
4. Write, in words, the sequence of Pandas operations you would use to find the county with the highest mean elevation among streams only.
5. Which Pandas operation from today do you think will be most useful for your own research data, and why?

## Optional extension if you finish early
Use `PRIM_LAT_DEC` and `PRIM_LONG_DEC` to make a simple scatter plot of Indiana feature locations. Color or separate the points by one feature class of your choice.

You may use Pandas' built-in `.plot.scatter()` or Matplotlib. This extension is intentionally **not scaffolded**: decide what subset you want to visualize and what question the figure should answer.